<div align="center">
    <img src="assets/neovim-logo.svg" width="260" class="no-zoom" alt="Neovim"/>
</div>

# Neovim Essentials: Day 3 - Plugins Showcase

On Day 03, we will take a look at some useful (and cool) plugins that help to get us to our "flow" state.

# Overview

Today, we will go through some useful plugins, and their keybindings that helps you in day-to-day programming work.

### Language Server Protocols

#### What is it?

The **Language Server Protocol (LSP)** is an open standard, created by Microsoft for VS Code, for how an editor talks to a **language server**: a separate program that understands one language deeply. The language server parses your project and answers questions like *"where is this function defined?"*, *"what can I type here?"* or *"what's wrong with this line?"*.

Before LSP, every editor had to implement "smart" features for every language itself: *M* editors × *N* languages = *M × N* integrations, most of them half-finished. With LSP, each language ships **one** server and each editor implements **one** client, so you only need *M + N* pieces of work. That is why Neovim gets the same Python intelligence as VS Code: they can run the *same* language server.

```
┌──────────────┐   "go to definition of `fetch`        ┌─────────────────┐
│   Neovim     │    at main.py line 42, column 8"       │ Language server │
│ (LSP client) │ ─────────────────────────────────────▶ │  e.g. pyright   │
│              │ ◀───────────────────────────────────── │                 │
└──────────────┘   "main.py, line 17, column 4"         └─────────────────┘
                     (JSON-RPC messages over stdio)
```

A few things worth knowing:

- **Neovim has a built-in LSP client** (since 0.5). It doesn't contain any language servers: you install those separately, one per language.
- **The server does the thinking, Neovim does the display.** Completions, diagnostics and hover docs all come from the server. If a feature is missing or wrong, it is usually the server (or its configuration), not Neovim.
- **LSP ≠ Treesitter.** Treesitter (Day 2) parses the *current file* to colour and select code. LSP understands the *whole project*: imports, types, references across files. You want both.
- **LSP is not a linter or formatter** (though some servers do both). Tools like `ruff`, `prettier` or `stylua` are often run alongside it, via plugins like `conform.nvim` and `nvim-lint` in LazyVim.

#### What it gives you

| Feature | What it means |
|---|---|
| Diagnostics | Errors and warnings shown inline as you type, without running the code |
| Completion | Suggestions that know the types of your variables and the members of your objects |
| Go to definition / references | Jump to where a symbol is defined, or list everywhere it's used, across files |
| Hover | Docs and the type signature of the symbol under the cursor |
| Rename | Rename a symbol everywhere it's used, safely (not a text search-and-replace) |
| Code actions | Quick fixes such as "add missing import" or "organise imports" |
| Formatting | Format the buffer, if the server supports it |

#### Installing language servers with Mason

Language servers are ordinary programs (`pyright`, `rust-analyzer`, `gopls`, `lua-language-server`, ...). You *could* install them with `brew`, `npm` or `pip`, but LazyVim comes with [Mason](https://github.com/mason-org/mason.nvim), a package manager for them that lives inside Neovim and installs everything into `~/.local/share/nvim/mason/`.

| Command / key | What it does |
|---|---|
| `:Mason` | Open the Mason window: everything available, and what's installed |
| `i` (in Mason) | Install the package under the cursor |
| `u` / `U` (in Mason) | Update the package under the cursor / update all |
| `X` (in Mason) | Uninstall the package under the cursor |
| `g?` (in Mason) | Show Mason's help |

The easiest route in LazyVim, though, is a **language extra**: run `:LazyExtras`, move to e.g. `lang.python`, and press `x` to enable it. The extra installs the right language server (and formatter, linter, Treesitter parser...) through Mason and configures it, so it all just works. Restart Neovim afterwards.

#### Using LSP in LazyVim

These work in any buffer with a language server attached:

| Key | What it does |
|---|---|
| `gd` | Go to definition |
| `gr` | List references |
| `gI` | Go to implementation |
| `gy` | Go to type definition |
| `K` | Hover: show docs for the symbol under the cursor (press again to jump into the popup) |
| `gK` | Signature help: the arguments of the function you are calling |
| `<Space>ca` | Code actions |
| `<Space>cr` | Rename the symbol under the cursor |
| `<Space>cf` | Format the buffer |
| `]d` / `[d` | Jump to the next / previous diagnostic |
| `<Space>cd` | Show the diagnostics for the current line |
| `<Space>xx` | Open all diagnostics in a list (Trouble) |
| `<Ctrl-o>` | Jump back to where you were before `gd`, `gr`, etc. |

> 💡 Press `<Space>c` and wait: which-key lists the code/LSP actions available in the current buffer.

Neovim (0.11+) also has some LSP keymaps built in, so they work even without LazyVim: `K` (hover), `grn` (rename), `gra` (code action), `grr` (references), `gri` (implementation), `[d` / `]d` (diagnostics) and `<Ctrl-s>` in Insert mode (signature help).

#### Is it working?

If a feature doesn't seem to do anything, check whether a server is attached to the current buffer:

| Command | What it does |
|---|---|
| `:checkhealth vim.lsp` | Show the attached language servers, their settings and any errors |
| `:LspInfo` | Same, on Neovim versions where nvim-lspconfig still provides it |
| `:LspRestart` | Restart the language server(s), e.g. after creating a virtual environment |
| `:Mason` | Check that the server is actually installed |

Common causes: the server isn't installed, the file type isn't recognised (`:set filetype?`), or the server can't find the project root (most look for a marker such as `.git`, `pyproject.toml` or `package.json`).

#### Configuring a server

In LazyVim, servers are configured through the `nvim-lspconfig` plugin spec. To add a server or change its settings, create a file in `lua/plugins/`, e.g. `~/.config/nvim/lua/plugins/lsp.lua`:

```lua
return {
  {
    "neovim/nvim-lspconfig",
    opts = {
      servers = {
        -- Install (via Mason) and enable the Lua language server
        lua_ls = {},
        -- Pass settings to pyright
        pyright = {
          settings = {
            python = {
              analysis = { typeCheckingMode = "strict" },
            },
          },
        },
      },
    },
  },
}
```

LazyVim merges this with its own defaults, so you only list what you want to add or change. The server names (`lua_ls`, `pyright`, ...) are the ones used by [nvim-lspconfig](https://github.com/neovim/nvim-lspconfig/blob/master/doc/configs.md).

<details>
<summary><b>Without LazyVim: plain Neovim 0.11+</b></summary>

Neovim 0.11 added `vim.lsp.config()` and `vim.lsp.enable()`, so a server can be set up in a few lines of `init.lua` (with the server already installed and on your `PATH`):

```lua
vim.lsp.config("pyright", {
  cmd = { "pyright-langserver", "--stdio" },
  filetypes = { "python" },
  root_markers = { "pyproject.toml", ".git" },
})
vim.lsp.enable("pyright")
```

Installing the [nvim-lspconfig](https://github.com/neovim/nvim-lspconfig) plugin gives you ready-made configs for hundreds of servers, so `vim.lsp.enable("pyright")` alone is enough.
</details>

> 💡 Try it now: open the `weather-check` project from Day 2 with `nvim .`, and open `main.py`. Put the cursor on a function call and press `gd` to jump to its definition, then `<Ctrl-o>` to come back. Press `K` over one of the `TypedDict`s, use `gr` to see where it's used, and rename a variable with `<Space>cr`. Finally, introduce a typo in a variable name and watch the diagnostic appear.

### Telescope

#### What is it?

[Telescope](https://github.com/nvim-telescope/telescope.nvim) is a **fuzzy finder**: one pop-up window that lets you search through *lists of things* by typing a few letters of what you want. Files, open buffers, lines of text across the project, help pages, keymaps, git commits, LSP symbols, diagnostics, colour schemes... they are all just lists, and Telescope gives every one of them the same interface.

"Fuzzy" means you don't need to type the exact name. `wcmain` will happily match `weather-check/main.py`: the letters only have to appear in order.

Every Telescope window has the same three parts:

```
┌─ Results ──────────────────────────┐┌─ Preview ──────────────────────────┐
│   day-02/day-02.ipynb              ││ import json                        │
│   day-01/weather-check/README.md   ││ from typing import TypedDict       │
│ > day-01/weather-check/main.py     ││                                    │
│                                    ││ class Weather(TypedDict):          │
├─ Find Files ───────────────────────┤│     temperature: float             │
│ > wcmain                     3 / 42 ││     ...                            │
└────────────────────────────────────┘└────────────────────────────────────┘
```

- **Prompt**: what you type. Telescope filters and ranks the results as you go.
- **Results**: the matching entries. The `>` marks the selected one.
- **Preview**: the contents of the selected entry (a file, a help page, a commit diff...).

Under the hood, a Telescope search is made of a **finder** (where the list comes from, e.g. `fd` or `rg`), a **sorter** (how matches are ranked) and a **previewer**. The ones you use day to day are called **pickers**.

> ⚠️ **Telescope vs. the Snacks picker in LazyVim.** Since LazyVim v14, the default picker is [`snacks.picker`](https://github.com/folke/snacks.nvim/blob/main/docs/picker.md), not Telescope. It looks and behaves almost the same, and **LazyVim uses the same keymaps for both**, so everything in the "Using it" table below works either way. To switch to Telescope, run `:LazyExtras`, move to `editor.telescope` and press `x`, then restart Neovim. You can check which one you have by pressing `<Space><Space>`: the window title says which picker opened.

#### Before you start: external tools

Telescope relies on two command-line tools for speed. Check them with `:checkhealth telescope`:

| Tool | Used for | Install (macOS) |
|---|---|---|
| [`ripgrep`](https://github.com/BurntSushi/ripgrep) (`rg`) | Searching text inside files (live grep) | `brew install ripgrep` |
| [`fd`](https://github.com/sharkdp/fd) | Finding files (faster than `find`, and respects `.gitignore`) | `brew install fd` |

LazyVim's Telescope extra also installs [`telescope-fzf-native`](https://github.com/nvim-telescope/telescope-fzf-native.nvim), a sorter written in C. It needs `make` and a C compiler to build (on macOS: `xcode-select --install`).

#### Using it in LazyVim

The most useful pickers, from anywhere in Neovim:

| Key | What it does |
|---|---|
| `<Space><Space>` | Find files in the project (root directory) |
| `<Space>/` | Grep: search text across the project |
| `<Space>,` | Switch between open buffers |
| `<Space>:` | Command history |
| `<Space>fr` | Recent files |
| `<Space>fF` | Find files in the current working directory (instead of the project root) |
| `<Space>fc` | Find a file in your Neovim config |
| `<Space>fg` | Find files tracked by git |
| `<Space>sw` | Search for the word under the cursor (or the visual selection) |
| `<Space>sb` | Search lines in the current buffer |
| `<Space>sh` | Search Neovim's help pages |
| `<Space>sk` | Search keymaps: *"what was the key for...?"* |
| `<Space>ss` | Go to an LSP symbol (function, class...) in the current file |
| `<Space>sd` | Diagnostics in the current buffer (`<Space>sD` for the whole workspace) |
| `<Space>gc` | Git commits (with the diff in the preview) |
| `<Space>uC` | Colour schemes, with a live preview |
| `<Space>sR` | Resume: reopen the last picker, with your query and selection |

> 💡 Press `<Space>s` or `<Space>f` and wait: which-key shows every *search* and *find* picker available.

#### Inside the picker

Telescope opens in **Insert mode** so you can type straight away. Press `<Esc>` to go to Normal mode inside the picker, where `j` / `k` move and `q` (or `<Esc>` again) closes it.

| Key (Insert mode) | What it does |
|---|---|
| `<Ctrl-n>` / `<Ctrl-p>` (or `<Down>` / `<Up>`) | Next / previous result |
| `<Enter>` | Open the selected entry |
| `<Ctrl-v>` / `<Ctrl-x>` / `<Ctrl-t>` | Open in a vertical split / horizontal split / new tab |
| `<Ctrl-u>` / `<Ctrl-d>` | Scroll the preview up / down |
| `<Tab>` | Mark the entry (multi-select) and move on |
| `<Ctrl-q>` | Send all results to the quickfix list (`<Alt-q>` sends only the marked ones) |
| `<Ctrl-/>` | Show all the keys available in this picker (`?` in Normal mode) |
| `<Ctrl-c>` | Close the picker |

LazyVim's Telescope extra adds a few more: `<Alt-h>` / `<Alt-i>` to include hidden / git-ignored files in a file search, `<Ctrl-Up>` / `<Ctrl-Down>` to cycle through your previous queries, and `<Ctrl-f>` / `<Ctrl-b>` to scroll the preview.

#### Search syntax

With the fzf sorter, the prompt understands a few extra tokens. Separate them with spaces to combine them:

| Token | Matches | Example |
|---|---|---|
| `word` | Fuzzy match | `wcmain` → `weather-check/main.py` |
| `'word` | Exact match | `'test` |
| `^word` | Starts with | `^day-02` |
| `word$` | Ends with | `.py$` |
| `!word` | Does not contain | `!ipynb` |
| `a \| b` | Either one | `.lua$ \| .py$` |

For example, `^day .md$ !summary` finds Markdown files under the `day-*` folders, skipping the summaries.

> 💡 **Grep, then filter.** `<Space>/` passes your query to `ripgrep`. When you have many hits, press `<Ctrl-q>` to send them all to the quickfix list, then step through them with `]q` / `[q`. This is a quick way to visit every place a string is used, even in files no language server understands.

#### Configuring Telescope

In LazyVim, configure it like any other plugin, with a file in `lua/plugins/`, e.g. `~/.config/nvim/lua/plugins/telescope.lua`:

```lua
return {
  {
    "nvim-telescope/telescope.nvim",
    -- Add your own keymaps
    keys = {
      {
        "<leader>fp",
        function()
          -- Find files in the folder where lazy.nvim keeps your plugins
          require("telescope.builtin").find_files({
            cwd = require("lazy.core.config").options.root,
          })
        end,
        desc = "Find Plugin File",
      },
    },
    -- Change options; LazyVim merges these with its own
    opts = {
      defaults = {
        layout_strategy = "horizontal",
        layout_config = { prompt_position = "top" },
        sorting_strategy = "ascending",
        file_ignore_patterns = { "%.ipynb_checkpoints/" },
      },
    },
  },
}
```

Every picker is just a Lua function in `telescope.builtin`, and they all accept the same kind of options table (`cwd`, `hidden`, `default_text`...). Run `:Telescope` on its own to browse the full list of built-in pickers, or `:help telescope.builtin` to read about them.

<details>
<summary><b>Without LazyVim: plain Neovim with lazy.nvim</b></summary>

```lua
{
  "nvim-telescope/telescope.nvim",
  dependencies = {
    "nvim-lua/plenary.nvim",
    { "nvim-telescope/telescope-fzf-native.nvim", build = "make" },
  },
  config = function()
    local telescope = require("telescope")
    telescope.setup({})
    telescope.load_extension("fzf")

    local builtin = require("telescope.builtin")
    vim.keymap.set("n", "<leader>ff", builtin.find_files, { desc = "Find files" })
    vim.keymap.set("n", "<leader>fg", builtin.live_grep, { desc = "Live grep" })
    vim.keymap.set("n", "<leader>fb", builtin.buffers, { desc = "Buffers" })
    vim.keymap.set("n", "<leader>fh", builtin.help_tags, { desc = "Help tags" })
  end,
}
```
</details>

> 💡 Try it now: open this repository with `nvim .`. Press `<Space><Space>` and type `wcmain` to open `weather-check/main.py`. Press `<Space>/`, search for `TypedDict`, and send the results to the quickfix list with `<Ctrl-q>`. Then press `<Space>sk` and type `telescope` (or `picker`) to find out which other pickers are bound to keys. Finally, try `<Space>uC` and scroll through the colour schemes to see the live preview.

### ripgrep (live grep)

#### What is it?

[ripgrep](https://github.com/BurntSushi/ripgrep) (the command is `rg`) is a command-line tool for searching text across files, like `grep -r`, only much faster and with better defaults. **It is not a Neovim plugin.** When you press `<Space>/`, the picker (Telescope or Snacks) runs `rg` in the background with whatever you type, and shows the matches as a list. The picker draws the window; ripgrep does the searching.

```
 you type "TypedDict"      ┌──────────────────┐   rg --vimgrep TypedDict   ┌──────────┐
 ────────────────────────▶ │ Picker           │ ─────────────────────────▶ │ ripgrep  │
                           │ (Telescope or    │ ◀───────────────────────── │  (rg)    │
 results + preview ◀────── │  Snacks)         │   main.py:3:19:from ...    └──────────┘
                           └──────────────────┘   main.py:7:15:class ...
```

The query is re-run on every keystroke, which is why it's called **live grep**. That only feels instant because ripgrep is so fast. It:

- **respects `.gitignore`**, so it skips `node_modules/`, `.venv/`, build output and so on;
- **skips hidden files and binary files** by default;
- **searches files in parallel**, using all your CPU cores;
- uses a fast **regular-expression** engine (Rust's `regex` crate).

Install it with `brew install ripgrep` (macOS) or `sudo apt install ripgrep` (Debian/Ubuntu), and check with `rg --version`. If `<Space>/` returns nothing at all, ripgrep is usually missing: run `:checkhealth telescope` or `:checkhealth snacks`.

#### Using it in LazyVim

| Key | What it does |
|---|---|
| `<Space>/` | Live grep in the project (root directory) |
| `<Space>sg` | Same as `<Space>/` |
| `<Space>sG` | Live grep in the current working directory (instead of the root) |
| `<Space>sw` | Grep for the word under the cursor (or the visual selection) |
| `<Space>sW` | Same, in the current working directory |
| `<Space>sr` | Search **and replace** across files (`grug-far.nvim`, which also runs ripgrep) |

Inside the results, the picker keys from the Telescope section all work: `<Enter>` to open, `<Ctrl-v>` for a split, `<Ctrl-q>` to send everything to the quickfix list.

#### The query is a regex

What you type is passed to ripgrep as a **regular expression**, so some characters have special meaning:

| You type | Matches |
|---|---|
| `fetch` | `fetch` anywhere, including `fetch_weather` and `prefetch` |
| `def \w+\(` | Any Python function definition |
| `TODO\|FIXME` | Either word (`\|` is the regex "or") |
| `get\(` | The literal text `get(`: escape `(`, `)`, `.`, `[`, `*`, `?`, `+`, `{` with `\` |
| `^import` | Lines that start with `import` |

LazyVim's pickers use **smart case**: an all-lowercase query is case-insensitive, and as soon as you type a capital letter it becomes case-sensitive.

> 💡 In the **Snacks** picker you can pass extra flags to ripgrep after `--`. For example, `fetch -- -g *.py` searches only Python files, and `fetch -- -t md` only Markdown. Telescope's plain `live_grep` doesn't support this; the [`telescope-live-grep-args`](https://github.com/nvim-telescope/telescope-live-grep-args.nvim) extension adds it.

#### ripgrep on the command line

`rg` is just as useful in a terminal (or Neovim's `:terminal`) as it is inside a picker:

| Command | What it does |
|---|---|
| `rg TypedDict` | Search the current directory, recursively |
| `rg -i weather` | Case-insensitive |
| `rg -S weather` | Smart case (as in LazyVim) |
| `rg -w main` | Whole words only (`main`, not `domain`) |
| `rg -F 'get('` | Fixed string: no regex, no escaping needed |
| `rg -t py fetch` | Only Python files (`rg --type-list` shows all types) |
| `rg -g '*.lua' -g '!tests/' opts` | Include / exclude files with globs |
| `rg -C 2 fetch` | Show 2 lines of context around each match |
| `rg -l fetch` | Only list the names of matching files |
| `rg --hidden fetch` | Also search hidden files (`.github/`, `.env`...) |
| `rg -u` / `-uu` / `-uuu` | Ignore `.gitignore` / plus hidden files / plus binary files |

To use some flags every time, put them in a file (one per line) and point the `RIPGREP_CONFIG_PATH` environment variable at it:

```sh
# ~/.ripgreprc
--smart-case
--hidden
--glob=!.git/
```

```sh
# in ~/.zshrc
export RIPGREP_CONFIG_PATH="$HOME/.ripgreprc"
```

Neovim picks this up too, since it runs the same `rg`.

#### Find and replace across the project

ripgrep only *finds*; Neovim does the replacing. Two ways to do it:

1. **With the quickfix list** (works in any Neovim):
   - press `<Space>/`, search for `old_name`, and press `<Ctrl-q>` to send the results to the quickfix list;
   - run `:cdo s/old_name/new_name/gc | update`. `:cdo` runs the substitution on every line in the list, `c` asks you to confirm each one, and `update` saves each file.
2. **With grug-far** (`<Space>sr`): a buffer where you type the search and the replacement and see every change before applying it.

> 💡 For renaming a *symbol* (a variable, function or class), prefer the LSP rename (`<Space>cr`) from the LSP section: it understands the code, so it won't touch unrelated text that happens to match.

Neovim's built-in `:grep` command also uses ripgrep when it is installed (Neovim 0.10+): `:grep TypedDict` fills the quickfix list directly, without opening a picker. Open it with `:copen`.

> 💡 Try it now: in this repository, press `<Space>/` and search for `TypedDict`, then `def \w+\(` to list every function. Put the cursor on a word in `main.py` and press `<Space>sw`. Then open a terminal and compare `rg -t py def` with `rg -l weather`. Finally, use `<Space>/` and `<Ctrl-q>` to send the matches for a variable name to the quickfix list, and step through them with `]q` / `[q`.